# buffer-copy_-inplace — ex1: update a BatchNorm running_mean buffer with copy_

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `buffer-copy_-inplace`. Running the final beacon cell reports progress against the `PyTorch: in-place buffer copy` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: in-place buffer copy` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`buffer-copy_-inplace`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "buffer-copy_-inplace"
DD_SUBTOPIC = "PyTorch: in-place buffer copy"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## in-place buffer copy_ — quick refresher

Modules like `BatchNorm` track running statistics (mean, variance) as **buffers** — tensors that are part of the module state but DO NOT receive gradients. The canonical update pattern is `running_buf.copy_(new_value)`:

```python
self.running_mean.copy_(
    (1 - momentum) * self.running_mean + momentum * batch_mean
)
```

Why `copy_` and not `=`?
- `self.running_mean = new_tensor` would REBIND the attribute, losing the registered-buffer link (so `.to(device)`, `.state_dict()`, etc. would stop tracking it).
- `self.running_mean.copy_(new_tensor)` writes into the existing storage. Identity (`id`) is preserved.

`copy_` accepts any broadcastable tensor and ignores requires_grad on the source — exactly what we want for non-differentiable state updates.

### Exercise 1 — update a BatchNorm running_mean buffer with copy_

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `tensor.copy_(other)` to update a running-mean buffer in place while preserving the buffer's storage identity (so registered-buffer links survive).
> Keywords: copy_, buffer, in-place, batchnorm, running-mean, identity
> ```

**KCs targeted:** `buffer-copy_-inplace`, `inplace-param-update`

Implement `update_running_mean(running_mean, batch_mean, momentum)`. This is the BatchNorm running-stats update, simplified to a single buffer.

**The math.** Exponential moving average:
`new_running = (1 - momentum) * running_mean + momentum * batch_mean`
(PyTorch's BatchNorm uses this exact form when `momentum=0.1`.)

**The critical mechanic.** You must write the new value into the EXISTING `running_mean` tensor in place using `.copy_()`. Do NOT:
- Reassign: `running_mean = ...` — the caller's reference is   unaffected (the buffer registration is lost in real nn.Module code).
- Use `*=` or `+=` separately and accumulate — pointless extra storage.
- Use `.data =` — works but bypasses `copy_`'s shape/dtype checks.

Use exactly: `running_mean.copy_(new_value)`. Return nothing (the function mutates `running_mean` in place).

The test asserts that `id(running_mean)` is preserved before/after the call — proof the buffer identity survived.

In [ ]:
def update_running_mean(running_mean: Tensor, batch_mean: Tensor, momentum: float) -> None:
    # Compute the new EMA value in a fresh tensor, then write it into
    # the existing running_mean's storage via copy_. This preserves
    # id(running_mean) — critical for nn.Module's registered-buffer link.
    new_value = (1 - momentum) * running_mean + momentum * batch_mean
    running_mean.copy_(new_value)


<details><summary>Solution</summary>

```python
def update_running_mean(running_mean: Tensor, batch_mean: Tensor, momentum: float) -> None:
    # Compute the new EMA value in a fresh tensor, then write it into
    # the existing running_mean's storage via copy_. This preserves
    # id(running_mean) — critical for nn.Module's registered-buffer link.
    new_value = (1 - momentum) * running_mean + momentum * batch_mean
    running_mean.copy_(new_value)
```

**Why `copy_` and not `=`.** In `nn.Module.register_buffer('name', tensor)`, PyTorch stores the tensor reference in `self._buffers['name']` AND aliases `self.name`. If you do `self.name = new_tensor`, you rebind the attribute but `_buffers['name']` still points to the OLD tensor. `.to(device)`, `.state_dict()`, `.load_state_dict()` all walk `_buffers` — they'll see stale data. `copy_` writes into the old tensor's storage, so both references see the update.

**Why `copy_` over `.data =`.** `running_mean.data = new_value` does work in PyTorch (and preserves identity) but skips `copy_`'s shape/dtype checks, masking bugs. PyTorch's own BatchNorm uses `copy_` (or `*=` / `+=` directly on running_mean) — never `.data =`.

**The EMA math.** When `momentum=0.1`, the running mean has effective half-life ≈ 7 batches. PyTorch chose 0.1 as the default for BatchNorm because it's slow enough to smooth across batches but fast enough to track training-time distribution shift.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()